In [1]:
from components import *
train_data = np.memmap("../data/tiny-stories-v2-gpt4-train_encoded.npy", dtype=np.int16, mode="r")
valid_data = np.memmap("../data/tiny-stories-v2-gpt4-valid_encoded.npy", dtype=np.int16, mode="r")


In [9]:
# model parameters
vocab_size = 10000
context_length = 128
d_model = 512
num_layers = 4
num_heads = 8
d_ff = d_model // 3 * 8 // 64 * 64
rope_theta = 10000

batch_size = 64
n_epochs = 1000

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# optimizer parameters
lr = 1e-3
weight_decay = 1e-5
betas = (0.9, 0.999)


lm = transformerLM(vocab_size, context_length, d_model, num_layers, num_heads, d_ff, rope_theta)
# lm = transformerLM_torch(vocab_size, context_length, d_model, num_layers, num_heads, d_ff, rope_theta)

lm.to(device)

optimizer = AdamW(lm.parameters(), lr=lr, weight_decay=weight_decay, betas=betas, eps=1e-6)


# load data
train_sampler = lambda: data_loader(train_data, context_length, batch_size, device)
valid_sampler = lambda: data_loader(valid_data, context_length, 10000, device)

In [10]:
def train_epoch():
    lm.train()
    x, y = train_sampler()

    optimizer.zero_grad()
    logits = lm(x)
    loss = cross_entropy_loss(logits, y).mean()
    loss.backward()
    optimizer.step()

    print(f"train loss: {loss.item():.4f}")

    return loss.detach().item()

In [11]:
from tqdm.notebook import tqdm

for _ in tqdm(range(n_epochs)):
    train_epoch()

  0%|          | 0/1000 [00:00<?, ?it/s]

train loss: 9.2487
train loss: 8.6440
train loss: 7.9816
train loss: 7.5371
train loss: 7.0660
train loss: 6.5773
train loss: 6.2016
train loss: 5.9016
train loss: 5.6034
train loss: 5.4829
train loss: 5.1970
train loss: 5.1044
train loss: 4.9018
train loss: 4.9532
train loss: 4.7140
train loss: 4.7844
train loss: 4.6725
train loss: 4.6615
train loss: 4.5284
train loss: 4.5193
train loss: 4.3990
train loss: 4.3683
train loss: 4.2814
train loss: 4.2809
train loss: 4.2663
train loss: 4.1812
train loss: 4.1394
train loss: 4.2362
train loss: 4.0646
train loss: 4.1295
train loss: 4.1696
train loss: 4.1575
train loss: 3.9895
train loss: 4.0807
train loss: 4.0101
train loss: 3.8533
train loss: 3.9020
train loss: 3.8758
train loss: 4.0103
train loss: 3.9078
train loss: 3.9056
train loss: 3.7838
train loss: 3.8634
train loss: 3.6511
train loss: 3.8494
train loss: 3.9028
train loss: 3.7495
train loss: 3.6740
train loss: 3.6614
train loss: 3.7768
train loss: 3.6464
train loss: 3.7978
train loss: 

In [29]:
def valid():
    lm.eval()
    with torch.no_grad():
        x, y = valid_sampler()
        mini_batch_size = 32
        total_loss = 0
        for i in range(0, x.shape[0], mini_batch_size):
            x_i, y_i = x[i:i+mini_batch_size], y[i:i+mini_batch_size]
            logits = lm(x_i)
            loss = cross_entropy_loss(logits, y_i).mean()
            total_loss += loss.item() * x_i.shape[0]
        return total_loss / x.shape[0]

In [30]:
valid()

1.9552574272155763

In [31]:
@torch.no_grad()
def mode_generate(lm: transformerLM, tokenizer, context, output_length, device, k=3):
    ctx_ids = tokenizer.encode(context)
    all_ids = torch.zeros(size=(len(ctx_ids)+output_length,), device=device, dtype=torch.int32)
    all_ids[:len(ctx_ids)] = torch.tensor(ctx_ids, device=device, dtype=torch.int32)
    for i in range(output_length):
        logits = lm(all_ids[:len(ctx_ids) + i])[-1, :]
        topk = torch.topk(logits, k)
        probs = torch.softmax(topk.values, dim=-1)
        idx = torch.multinomial(probs, num_samples=1)
        all_ids[len(ctx_ids) + i] = topk.indices[idx]
    return tokenizer.decode(all_ids.cpu().numpy())

In [32]:
tokenizer_path = "../tokenizers/tiny-stories-v2-gpt4-train_tokenizer.pt"
tokenizer = torch.load(tokenizer_path, weights_only=False)

context = "there was once a little girl"
x = mode_generate(lm, tokenizer, context, 100, device)

In [33]:
print(x)

there was once a little girl. She had a pet cat named Luna.
One day, Tom and Luna were playing with a ball. Tom said, "I want to play with the ball!" Luna said, "No, I am not nice. I will play with the ball." They both felt sad.
Then, Tom had an idea. He said to Luna, "Let's make a ball!" They started to play with the ball. They rolled it, rolled it, and laughed together. They were happy to
